In [1]:

import os
import pickle
import mediapipe as mp
import cv2

mp_hands = mp.solutions.hands

hands = mp_hands.Hands(
    static_image_mode=True,
    min_detection_confidence=0.5,
    max_num_hands=2
)

VERSION = 2
DATA_DIR = f'./datas/data_v{VERSION}'

data = []
labels = []

for dir_ in os.listdir(DATA_DIR):

    for img_path in os.listdir(os.path.join(DATA_DIR, dir_)):

        data_aux = []

        img = cv2.imread(
            os.path.join(DATA_DIR, dir_, img_path)
        )

        if img is None:
            continue

        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        results = hands.process(img_rgb)

        if results.multi_hand_landmarks:

            # Store coordinates separately for each hand
            hand_data = {
                "Right": None,
                "Left": None
            }

            all_x = []
            all_y = []

            # Extract landmarks and handedness
            for i, hand_landmarks in enumerate(results.multi_hand_landmarks):

                mp_label = results.multi_handedness[i].classification[0].label

                hand_label = ("Right" if mp_label == "Left" else "Left")

                coords = []

                for landmark in hand_landmarks.landmark:

                    x = landmark.x
                    y = landmark.y

                    coords.append((x, y))

                    all_x.append(x)
                    all_y.append(y)

                # Assign landmarks to the correct hand
                hand_data[hand_label] = coords

            # Shared normalization for both hands
            min_x = min(all_x)
            min_y = min(all_y)

            width = max(all_x) - min_x
            height = max(all_y) - min_y

            scale = max(width, height)

            if scale == 0:
                continue

            # Always extract Right first, then Left
            for hand_label in ["Right", "Left"]:

                if hand_data[hand_label] is not None:

                    for x, y in hand_data[hand_label]:

                        normalized_x = (x - min_x) / scale
                        normalized_y = (y - min_y) / scale

                        data_aux.append(normalized_x)
                        data_aux.append(normalized_y)

                else:
                    # Missing hand = 42 zero features
                    data_aux.extend([0.0] * 42)

            # Every sample must contain 84 features
            if len(data_aux) == 84:
                data.append(data_aux)
                labels.append(dir_)
                print(f"Saved {dir_}: {len(data_aux)} features")
            else:
                print("Invalid feature count:", len(data_aux))

        else:
            print(f"Skipping image: {dir_}/{img_path}"," - No hands detected")

# Save dataset
DATA_VER_PICKLE=3
with open(f'./datas/data_v{DATA_VER_PICKLE}.pickle', 'wb') as f:

    pickle.dump({
        'data': data,
        'labels': labels
    }, f)

hands.close()
print("Total samples:", len(data))

Saved 0: 84 features
Saved 0: 84 features
Saved 0: 84 features
Saved 0: 84 features
Saved 0: 84 features
Saved 0: 84 features
Saved 0: 84 features
Saved 0: 84 features
Saved 0: 84 features
Saved 0: 84 features
Saved 0: 84 features
Saved 0: 84 features
Saved 0: 84 features
Saved 0: 84 features
Saved 0: 84 features
Saved 0: 84 features
Saved 0: 84 features
Saved 0: 84 features
Saved 0: 84 features
Saved 0: 84 features
Saved 0: 84 features
Saved 0: 84 features
Saved 0: 84 features
Saved 0: 84 features
Saved 0: 84 features
Saved 0: 84 features
Saved 0: 84 features
Saved 0: 84 features
Saved 0: 84 features
Saved 0: 84 features
Saved 0: 84 features
Saved 0: 84 features
Saved 0: 84 features
Saved 0: 84 features
Saved 0: 84 features
Saved 0: 84 features
Saved 0: 84 features
Saved 0: 84 features
Saved 0: 84 features
Saved 0: 84 features
Saved 0: 84 features
Saved 0: 84 features
Saved 0: 84 features
Saved 0: 84 features
Saved 0: 84 features
Saved 0: 84 features
Saved 0: 84 features
Saved 0: 84 f